In [ ]:
! pip install numpy torch matplotlib PyYAML scipy opencv-python torchvision

In [ ]:
%load_ext autoreload
%autoreload 2

# Prepare SEA-RAFT

In [ ]:
import sys
sys.path.append('core')
import numpy as np
from  argparse import Namespace
import torch
import matplotlib.pyplot as plt
from torchvision.io import read_image
from torchvision.transforms import Resize
%env CUDA_VISIBLE_DEVICES=4
from tqdm import tqdm
config = Namespace(**{

    "name": "MMFI-custom",
    "dataset": "MMFI",
    "gpus": [0],
    "use_var": True,
    "var_min": 0,
    "var_max": 10,
    "pretrain": "resnet34",
    "initial_dim": 64,
    "block_dims": [64, 128, 256],
    "radius": 4,
    "dim": 128,
    "num_blocks": 2,
    "iters": 12,
    "image_size": [256,336], #[128,168],
    "scale": 0,
    "batch_size": 16,
    "epsilon": 1e-8,
    "wdecay": 1e-5,
    "dropout": 0,
    "clip": 1.0,
    "gamma": 0.85,
    "num_steps": 10000,
    "restore_ckpt": None,
    "coarse_config": None,
    "model":"searaft_weights/Tartan-C-T-TSKH432x960-M.pth"

})

resizer = Resize((config.image_size))
#gray = Grayscale(3) ## TODO this is very rudimerory
#model = RAFT(config)
#load_ckpt(model, config.model)
image1 = resizer(read_image("dataset/wimans/E01/S01/A01/image/frame0003.png")[None])
image2 = resizer(read_image("dataset/wimans/E01/S01/A01/image/frame0004.png")[None])

#model.eval()

device = "cuda"
#model.to(device)
image1 = image1.to(device)
image2 = image2.to(device)
image1.shape, image2.shape


In [ ]:
from pathlib import Path
from viz import my_flow_to_image, plot_image
output = model(image1, image2, iters=config.iters, test_mode=True)  # noqa: F821
flow_img = my_flow_to_image(output["final"].to("cpu"))
output["final"].shape
plot_image(image1.to("cpu"),image2.to("cpu"),output["final"].to("cpu"))


In [ ]:
import os
import importlib
from mmfi_dataset.decode_config import MMFIConfig
from mmfi_dataset.mmfi import  MMFi_Database
from WIFlow.dataset import collate_fn_padd
from torch.utils.data import DataLoader
from torchvision import transforms
from WIFlow import dataset
# Please add the downloaded mmfi directory into your python project.
rng_generator = torch.manual_seed(42)
importlib.reload(dataset)
dataset_config = MMFIConfig.load('dataset_configs/wimans_config_E1_all.yml')
database = MMFi_Database(dataset_config.dataset_root)
database.scenes

train_dataset = dataset.MMFI_DatasetPairwise(database,dataset_config.modalities,dataset_config.train) 
resizer = transforms.Resize(config.image_size)
rng_generator = torch.manual_seed(dataset_config.seed)
all_loader = DataLoader(
    train_dataset,
    batch_size=dataset_config.train.batch_size,
    collate_fn=collate_fn_padd,
    num_workers=4,
    shuffle=False,
    drop_last=False,
    generator=rng_generator,
    )
len(all_loader)


# Generate Pseudo-GT-FLow based on sea-raft

In [ ]:
for X1,X2 in tqdm(all_loader):
    target_path_dir = Path(X1["image_path"][0][0]).parent.parent/"flow"
    if not target_path_dir.is_dir():
        os.mkdir(target_path_dir)

    frame_i = 1
    im1 , im2 = resizer(X1["image"].to(device)), resizer(X2["image"].to(device))
    output = model(im1, im2, iters=config.iters, test_mode=False)  # noqa: F821
    for i, flow in enumerate(output["final"]):
        frame_name = X1["image_path"][0][0].name.split(".")[0]
        flow_path = target_path_dir/f"{frame_name}.pt"
        print(flow.shape)
        print(len(output["final"]))
        break
#        torch.save(flow.to(torch.float16), flow_path)


# Generate Classes modalitly

In [ ]:
import re
import json
from collections import defaultdict
pattern = re.compile(r"(_\d+Mhz|_\d+)+$")
pattern = re.compile(r"[^_]+_[^_]+_([a-zA-Z_]+)_(?=\d{2})")
distribution = defaultdict(lambda:0)

s = "void_speed_40Mhz_3"
cleaned = pattern.sub("", s)

for X1,X2 in tqdm(all_loader):
    target_path_dir = Path(X1["image_path"][0][0]).parent.parent/"class"
    if not target_path_dir.is_dir():
        os.mkdir(target_path_dir)
    with open(Path(X1["image_path"][0][0]).parent.parent/"action_meta.json") as f:
        raw = json.load(f)["action_name"]
        cls = pattern.findall(raw)[0]
    frame_name = X1["image_path"][0][0].name.split(".")[0]
    cls_path = target_path_dir/f"{frame_name}.txt"
    distribution[cls]+=1
    cls_path.write_text(cls, encoding="utf-8")
distribution

In [ ]:
distribution

# Generate pseudo GT based on multiple models
- SEARAFT
- MemFlow
- DPFlow


In [ ]:
import torch
import ptlflow
from ptlflow.utils.io_adapter import IOAdapter
from torchvision.io import read_image
device = "cuda:0"
# Get an optical flow model. As as example, we will use RAFT Small
# with the weights pretrained on the FlyingThings3D dataset
model_configs = [
    #("raft", "things")
    ("rpknet", "things"),
    ("ms_raft_p", "mixed"),
    ("sea_raft_m", "spring"), # kitti does match to old by docu but results are different
    ("memflow", "spring"),
    ("dpflow", "spring"),
]

model_configs = [
    ("rpknet", "things"),
    ("ms_raft_p", "mixed"),
    ("sea_raft_m", "spring"),
    ("memflow", "spring"),
    ("dpflow", "spring"),
]
models = [ptlflow.get_model(name, ckpt_path=check).to(device).eval() for name, check in model_configs]

# Load the images

images = [
    image1.cpu()[0].permute(1,2,0),
    image2.cpu()[0].permute(1,2,0),
]

# Stack images into a list
io_adapters = [IOAdapter(model, images[0].shape[:2],target_size=config.image_size, cuda=True,device=device) for model in models]

# inputs is a dict {'images': torch.Tensor}
# The tensor is 5D with a shape BNCHW. In this case, it will have the shape:
inputs = [io_adapter.prepare_inputs(images) for io_adapter in io_adapters]
# Forward the inputs through the model
flows_dict = [model(input) for model,input in zip(models,inputs)]



# flows will be a 5D tensor BNCHW.
# This example should print a shape (1, 1, 2, H, W).
print(flows_dict[0]["flows"].shape)


In [ ]:
from viz import my_flow_to_image
flows = torch.concat([flow["flows"][0] for flow in flows_dict])
#flows = torch.flip(flows, dims=[1])

flows_rgb = my_flow_to_image(flows)
fig, axs = plt.subplots(1, len(flows), figsize=(6 * len(flows), 6))

for ax, model, flow_rgb in zip(axs,models,flows_rgb):
    flow_rgb = flow_rgb.permute(1, 2, 0)
    flow_rgb_npy = flow_rgb.detach().cpu().numpy()
    ax.imshow(flow_rgb)
    ax.set_title(model.__class__.__name__)
    ax.axis('off')
plt.tight_layout()
plt.show()
fig_img, axs_img = plt.subplots(1, 2, figsize=(12, 6))
axs_img[0].imshow(images[0])
axs_img[0].set_title("Image 1")
axs_img[0].axis('off')
axs_img[1].imshow(images[1])
axs_img[1].set_title("Image 2")
axs_img[1].axis('off')
plt.tight_layout()
plt.show()


In [ ]:
for X1,X2 in tqdm(all_loader):
    target_path_dir = Path(X1["image_path"][0][0]).parent.parent/"flow"

    if not target_path_dir.is_dir():
        os.mkdir(target_path_dir)
    frame_name = X1["image_path"][0][0].name.split(".")[0]
    flow_path = target_path_dir/f"{frame_name}.pt"
    if flow_path.is_file():
        continue
    im1 , im2 = resizer(X1["image"].to(device)), resizer(X2["image"].to(device))
    images = [
        im1.cpu()[0].permute(1,2,0),
        im2.cpu()[0].permute(1,2,0), 
    ]
    inputs = [io_adapter.prepare_inputs(images) for io_adapter in io_adapters]
    # Forward the inputs through the model
    flows_dict = [model(input) for model,input in zip(models,inputs)]
    flows = torch.concat([flow["flows"][0] for flow in flows_dict])


    avg_flow = torch.mean(flows, dim=0)

    flow_avg_amp = (avg_flow**2).sum(dim=0, keepdim=True).sqrt()
    mask_amp = ~(flow_avg_amp > 0.5).to(torch.bool)

    
    with torch.no_grad():
        avg_flow[0][mask_amp.squeeze(0)] = 0.0
        avg_flow[1][mask_amp.squeeze(0)] = 0.0
    # flows_rgb = my_flow_to_image(flows)
    # fig, axs = plt.subplots(1, len(flows)+3, figsize=(6 * len(flows), 6))
    # for ax, model, flow_rgb in zip(axs,models,flows_rgb):
    #     flow_rgb = flow_rgb.permute(1, 2, 0)
    #     flow_rgb_npy = flow_rgb.detach().cpu().numpy()
    #     ax.imshow(flow_rgb)
    #     ax.set_title(model.__class__.__name__)
    #     ax.axis('off')
    # avg_flow_rgb = my_flow_to_image(avg_flow[None])[0].permute(1,2,0).detach().cpu().numpy()
    # axs[-3].imshow(avg_flow_rgb)
    # axs[-3].set_title("Averge Flow")
    # axs[-3].axis('off')

    # axs[-2].imshow(images[0])
    # axs[-2].set_title("Image 1")
    # axs[-2].axis('off')
    # axs[-1].imshow(images[1])
    # axs[-1].set_title("Image 2")
    # axs[-1].axis('off')
    # plt.tight_layout()
    # plt.show()
    

    torch.save(avg_flow.cpu().detach(), flow_path)


# fix E06 to have good complex CSI instead of real = amplitude and .imag = phase



In [ ]:
def to_true_complex(csi_old:torch.Tensor)-> torch.Tensor:
    return csi_old.real * np.exp(1j * csi_old.imag)
for X1,X2 in tqdm(all_loader):
    target_path = Path(X1["csi_path"][0][0])
    true_csi_comp:torch.Tensor = to_true_complex(X1["csi"])
    #torch.save(true_csi_comp.cpu().detach(), target_path)